<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Let callers say what they need

**[Let callers say what they need](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/)** · Speech AI: Transcribe, Understand and Speak · Module 7, lesson 4 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a caller's spoken answer into an intent with keyword rules, measure the rules on the recorded answers of the IVR test calls, find the answers they miss, and make them better without breaking the answers that worked.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The keypad menu and its misses, from [A keypad menu first](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M07-L04-let-callers-say-what-they-need/let-callers-say-what-they-need-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/ivr/calls.jsonl": "5ee1fbad92c79b73fa908076325f2ce6d4030ed12439dacc26b6dcd9bc00d828",
    "data/recordings/agent/inputs.jsonl": "4a9274d0f9dab006ca40634525c761c27afc75d9f5b9f889501d5debe0c4c5d1",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The menu answers

The IVR asks "How can I help?". Each IVR test call holds the caller's answer as a recording. The file `inputs.jsonl` holds what **Whisper base** wrote for each caller turn (with no state prompt). Run the next cell: it collects the spoken answers to the menu, with what the caller said, what Whisper wrote, and the intent the test expects.

In [ ]:
ivr_calls = {}
for line in open("data/ivr/calls.jsonl", encoding="utf-8"):
    row = json.loads(line)
    ivr_calls[row["call_id"]] = row
recognised = {}
for line in open("data/recordings/agent/inputs.jsonl", encoding="utf-8"):
    row = json.loads(line)
    recognised[row["call_id"]] = row

# The caller's answer to the menu: the turn after the language turn, when it is spoken (not keys, not a number).
answers = []
for cid, call in ivr_calls.items():
    turn = call["inputs"][1] if len(call["inputs"]) > 1 else {}
    if "say" not in turn or turn.get("spoken"):
        continue
    answers.append({"call": cid, "kind": call["kind"], "language": call["language"], "said": turn["say"],
                    "heard": recognised[cid]["turns"][1]["text"], "intent": call["expected"].get("intent")})
print(len(answers), "spoken menu answers")
for a in answers[:5]:
    print(f'{a["call"]} {a["language"]}  said: {a["said"]!r:45} Whisper base: {a["heard"]!r}  expected: {a["intent"]}')

> **Check.** 45 spoken menu answers. An unclear request (`None`) expects no intent: the right answer there is "no match".

## 2. The IVR's keyword rules

These are the rules of the Larkfield IVR (`larkspeech/ivr/dialogue.py`). Each intent has a list of phrases. The text is normalised first (lower case, no punctuation), and **the last intent the caller names wins**.

> **Predict.** The rules on the callers' own words, and on Whisper's text: how many of the 45 answers will each get right?

In [ ]:
INTENT_WORDS = {
    "order_status": ["order status", "where is my order", "track", "tracking", "delivery", "hasn't arrived",
                     "has not arrived", "suivi", "suivre", "où est ma commande", "où est la commande", "livraison"],
    "opening_hours": ["opening hours", "hours", "when are you open", "open", "heures", "ouverture", "ouverts",
                      "ouvert"],
    "start_return": ["return", "send back", "send it back", "retour", "retourner", "renvoyer"],
    "agent": ["person", "agent", "someone", "human", "representative", "quelqu'un", "quelqu un", "personne",
              "agent", "conseiller"],
}

def find_intent(text, language, words=INTENT_WORDS):
    """Keyword rules. The LAST intent the caller names wins."""
    t = " " + cp.normalise(text, "basic", language) + " "
    best, pos = None, -1
    for intent, phrases in words.items():
        for w in phrases:
            i = t.rfind(" " + cp.normalise(w, "basic", language) + " ")
            if i > pos:
                best, pos = intent, i
    return best

def score(words, which):
    """How many menu answers get the expected intent (None is right for an unclear request)."""
    right = [a["call"] for a in answers if find_intent(a[which], a["language"], words) == a["intent"]]
    return right

on_said = score(INTENT_WORDS, "said")
on_heard = score(INTENT_WORDS, "heard")
print(f"rules on what the callers said: {len(on_said)} of {len(answers)} right")
print(f"rules on what Whisper base wrote: {len(on_heard)} of {len(answers)} right")

> **Check.** On the callers' own words: 45 of 45. On Whisper base's text: 35 of 45. Recognition errors cost some answers, and the rules miss others even on the right words.

## 3. Read the misses

In [ ]:
for a in answers:
    got = find_intent(a["heard"], a["language"])
    if got != a["intent"]:
        print(f'{a["call"]} {a["language"]}  heard {a["heard"]!r:50} -> {got}   expected {a["intent"]}')

Some misses are recognition errors that no rule can fix: "Start your attendees." for "Start a return, please." Others are close: "Je suis vide ma commande." for "Le suivi de ma commande." still has the word *commande*. And one is a contraction that the rules do not know.

## 4. By language

In [ ]:
for lang in ("en", "fr"):
    n = sum(a["language"] == lang for a in answers)
    ok = sum(1 for a in answers if a["language"] == lang and a["call"] in on_heard)
    print(f"{lang}: {ok} of {n} right on Whisper base's text")

> **Check.** English: 18 of 20. French: 17 of 25.

## 5. Change one thing: a contraction

> **Predict.** The rules know "where is my order". Do they know "Where's my order?"

In [ ]:
print(repr(cp.normalise("Where's my order?", "basic", "en")))
print(find_intent("Where's my order?", "en"), "|", find_intent("Where is my order?", "en"))

> **Check.** After normalisation, "Where's" becomes `where s`, so the phrase "where is my order" does not match, and no other phrase does either: no intent. Test call I-006 failed this way.

## 6. Your turn: better rules, nothing broken

> **Your turn.** Add phrases to `MY_WORDS` so that more answers on Whisper's text get the right intent. Write the phrases as they look **after** normalisation (lower case, no apostrophe: `where s my order`). Then run the next cell. A good change makes some answers newly right and breaks none.

In [ ]:
import copy
MY_WORDS = copy.deepcopy(INTENT_WORDS)
MY_WORDS["order_status"] += ["where s my order", "commande"]
MY_WORDS["opening_hours"] += []
MY_WORDS["start_return"] += []
MY_WORDS["agent"] += []

In [ ]:
mine = score(MY_WORDS, "heard")
newly_right = sorted(set(mine) - set(on_heard))
broken = sorted(set(on_heard) - set(mine))
print(f"{len(mine)} of {len(answers)} right (before: {len(on_heard)}); newly right: {newly_right}; broken: {broken}")
print("on the callers' own words:", len(score(MY_WORDS, "said")), "of", len(answers))

In [ ]:
expect("more answers right than before", len(mine) > len(on_heard), True)
expect("answers that you broke", len(broken), 0)

> **Check.** The solution adds `where s my order` and the French word `commande`: 39 of 45 right (newly right: I-005, I-008, I-011, I-052), none broken. A short word like `commande` is risky: "Je veux faire un retour pour ma commande" names a return first and *commande* last, so the last-intent rule would now say `order_status`. Rules that look better on one test set can be worse on the next callers. Run the cell of section 3 with `MY_WORDS` to see what is left.

## 7. Failure case: a word with two meanings

In [ ]:
print(find_intent("I want to send it back, it does not work.", "en"))
print(find_intent("Can I return to the menu?", "en"))

> **Check.** Both return `start_return`. The first is right. The second caller wants the **menu** again, not a return. Keyword rules cannot tell the two apart: the confirmation step and the way to a person must catch such mistakes. A classifier trained on many real menu answers could learn the difference, but Larkfield does not have those answers yet.

## Next

You measured keyword rules on recorded answers, found the misses, and improved the rules without breaking what worked. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/) has the end of turn, the state prompts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Numbers, confirmation and recovery](https://learning.nextia-ai.com/courses/speech/m07/numbers-confirmation-and-recovery/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.